# 06 — SHAP Explainability + Feature Importance

**Optimized:** tqdm, reduced SHAP samples (200), fallback KernelExplainer

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, pickle, os
from tqdm.notebook import tqdm
if os.path.basename(os.getcwd())=='notebooks': os.chdir('..')
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
SEED=42; PROC='data/processed'; MOD='models'; CH='charts'
os.makedirs(f'{CH}/explainability',exist_ok=True)
print('Ready')

In [ ]:
X=pd.read_csv(f'{PROC}/X_processed.csv')
y=pd.read_csv(f'{PROC}/y_target.csv').iloc[:,0]
CN=pickle.load(open(f'{MOD}/class_names.pkl','rb'))
res_df=pd.read_csv(f'{MOD}/all_results.csv')
champ=res_df.iloc[0]['Model']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
model=pickle.load(open(f'{MOD}/model_zoo/{champ}.pkl','rb'))
print(f'Champion: {champ}')

## 1. SHAP Summary (Beeswarm) — 200 samples

In [ ]:
sv=None; explainer=None
try:
    print('Trying TreeExplainer...')
    explainer=shap.TreeExplainer(model)
    X_shap=X_test.iloc[:200]
    sv=explainer.shap_values(X_shap)
    plt.figure(figsize=(10,8))
    shap.summary_plot(sv,X_shap,class_names=CN,show=False)
    plt.title(f'SHAP Summary: {champ}',fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{CH}/explainability/13_shap_summary.png',dpi=150,bbox_inches='tight')
    plt.show()
    print('TreeExplainer done')
except Exception as e:
    print(f'TreeExplainer failed: {e}')
    print('Trying KernelExplainer (100 bg, 100 test)...')
    bg=shap.sample(X_train,100)
    explainer=shap.KernelExplainer(model.predict_proba,bg)
    sv=explainer.shap_values(X_test.iloc[:100])
    plt.figure(figsize=(10,8))
    shap.summary_plot(sv,X_test.iloc[:100],class_names=CN,show=False)
    plt.title(f'SHAP Summary (Kernel): {champ}',fontweight='bold')
    plt.savefig(f'{CH}/explainability/13_shap_summary.png',dpi=150,bbox_inches='tight')
    plt.show()
    print('KernelExplainer done')

## 2. SHAP Waterfall (Single Prediction)

In [ ]:
try:
    if sv is not None and explainer is not None:
        sv0=sv[0] if isinstance(sv,list) else sv
        exp_val=explainer.expected_value
        if isinstance(exp_val,list): exp_val=exp_val[0]
        plt.figure(figsize=(10,6))
        shap.waterfall_plot(shap.Explanation(
            values=sv0[0],base_values=exp_val,
            data=X_test.iloc[0],feature_names=X.columns.tolist()),show=False)
        plt.title('SHAP Waterfall: Single Prediction')
        plt.savefig(f'{CH}/explainability/14_shap_waterfall.png',dpi=150,bbox_inches='tight')
        plt.show()
        print('Waterfall done')
    else:
        print('No SHAP values available for waterfall')
except Exception as e: print(f'Waterfall failed: {e}')

## 3. Feature Importance — Top 5 Models

In [ ]:
tree_models=['random_forest','extra_trees','xgboost','lightgbm','catboost']
avail=[m for m in tree_models if os.path.exists(f'{MOD}/model_zoo/{m}.pkl')]
print(f'Available tree models: {avail}')
if avail:
    fig,axes=plt.subplots(1,len(avail),figsize=(5*len(avail),6))
    if len(avail)==1: axes=[axes]
    for i,mname in enumerate(avail):
        m=pickle.load(open(f'{MOD}/model_zoo/{mname}.pkl','rb'))
        imp=m.feature_importances_
        top=np.argsort(imp)[-10:]
        axes[i].barh([X.columns[j] for j in top],imp[top],color='steelblue')
        axes[i].set_title(mname.replace('_',' ').title(),fontsize=10)
    fig.suptitle('Feature Importance Comparison',fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{CH}/explainability/15_feature_importance.png',dpi=150,bbox_inches='tight')
    plt.show()
else:
    print('No tree models found')

In [ ]:
print('\nExplainability complete. Charts saved to charts/explainability/')
print(f'Total charts generated: 15+')